In [125]:
import fitz  # Import the PyMuPDF library
import pandas as pd
pd.set_option('mode.copy_on_write', True)
import re
import os
import glob
import spacy
nlp = spacy.load("en_core_web_sm") 
from spacy.matcher import Matcher
lemmatizer = nlp.get_pipe("lemmatizer")
import datetime
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
import json
import numpy as np
import seaborn as sns
from scipy.spatial import distance
from scipy.stats import ttest_rel
import matplotlib.pyplot as plt
import statsmodels.stats.multitest as sm
import pingouin as pg
from math import sqrt

In [4]:
#Import Civil Service Job Adverts
# Path to the folder containing CSV files
path = "/Users/markrichardson-griffiths/Documents/Masters/Capstone/Data/Civil Service Jobs/"

# List all files in the folder
files = os.listdir(path)

# Initialize an empty list to store references to each batch of jobs batches (DataFrames)
job_batches = []

# Loop through each CSV file and read it into a DataFrame
for file in files:
    if file in ".DS_Store":
        continue
    else:
        file_path = path + file
        df_job_batch = pd.read_csv(file_path)
        job_batches.append(df_job_batch)

# Concatenate all job batches
df_all_jobs = pd.concat(job_batches, ignore_index=True)

In [5]:
#For info. count number of vacancies
print(len(df_all_jobs))

6373


In [6]:
#Reusable function to get most recent file

def identify_most_recent_file(path, base):
  
    pattern = os.path.join(path, f'{base}*.csv')
    csv_files = glob.glob(pattern)
    latest_file = max(csv_files, key=os.path.getmtime)
    return latest_file


In [7]:
#Import literature and store for pre-processing

path = "/Users/markrichardson-griffiths/Documents/Masters/Capstone/Data/Expert_Corpus/Consolidated/"
base = 'consolidated_lit'
file = identify_most_recent_file(path, base)
df_lit_docs = pd.read_csv(file) 

In [8]:
#Carry out preprocessing

#Remove duplicate jobs
df_unique_jobs = df_all_jobs.drop_duplicates(subset=['ref_no'], keep='last')

#Remove non-white space, non-characters and formatting such as /n from jobs
def remove_noise(text):
     return re.sub(r'[^\\w\\s]', '', text).replace('\\n', ' ').replace('\\xa0', ' ').replace('About the job Job', '').replace('summary', '')

df_unique_jobs['about_the_job'] = df_unique_jobs['about_the_job'].apply(remove_noise)

In [9]:
#For info - count unique jobs/vacancies
print(f"The number of jobs being analysed is: {len(df_unique_jobs)}")

The number of jobs being analysed is: 2051
